In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.optimize
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.special import logsumexp
import warnings
from scipy.optimize import curve_fit
from scipy.optimize import approx_fprime
import scipy.stats as stats
from pybads import BADS

from matplotlib.colors import LinearSegmentedColormap

# TEMP
warnings.simplefilter("ignore", category=FutureWarning)

from src.src import *

# Set some defaults
plt.rc("axes.spines", top=False, right=False)
plt.rcParams["mathtext.default"] = "regular"
sns.set_theme(font_scale=1.2)
sns.set_style("ticks")
# sns.set_palette('deep')

%config InlineBackend.figure_format = "retina"

## The Plan

### Parameter and model recovery 
1) Simulate data using best-fit parameters from each model (or samples from within the range of best-fits). For example, 100 simulations of each model within range of best-fit parameters.
2) Refit simulated data using MLE.
3) Correlate parameters used for simulation with recovered parameters.
4) Model comparison with AIC/BIC. 

In [2]:
# Read in behavioral data
sdt = pd.read_csv("../behavioural/data/processed-sdt.csv")
adapt = pd.read_csv("../behavioural/data/processed-adapt.csv")

# Read in csv with MLEs from all models
# Need a function to convert string back to numpy array
def converter(input_str):
    return np.fromstring(input_str[1:-1], sep=' ')

fits_sdt = pd.read_csv("model-fits/fits_sdt.csv", converters={"params":converter})
fits_adapt = pd.read_csv("model-fits/fits_adapt.csv", converters={"params":converter})
fits_sdt.head()


,SN,model,fit_task,nll,aic,bic,params,C,sigma_v,b,sigma_comb,alpha,beta,sigma_p,sigma_pert,aic_weighted,bic_weighted
0,1,vco,sdt,318.538186,643.076371,657.122697,"[3.41250859, 1.82921363, -0.11052107]",3.412509,1.829214,-0.110521,NaN,NaN,NaN,NaN,NaN,935.269293,950.448146
1,2,vco,sdt,286.700472,579.400945,593.447271,"[5.1097239, 2.46265332, 2.78463823]",5.109724,2.462653,2.784638,NaN,NaN,NaN,NaN,NaN,843.107896,858.289325
2,3,vco,sdt,466.611546,939.223091,953.189412,"[3.46698499, 3.8885529, 0.20789672]",3.466985,3.888553,0.207897,NaN,NaN,NaN,NaN,NaN,1309.749891,1324.719279
3,4,vco,sdt,367.046285,740.092570,754.142653,"[4.10714784, 2.64430312, 0.25845051]",4.107148,2.644303,0.258451,NaN,NaN,NaN,NaN,NaN,1071.766434,1086.934960
4,5,vco,sdt,266.566256,539.132513,553.178838,"[4.82265147, 2.26528203, 0.07786751]",4.822651,2.265282,0.077868,NaN,NaN,NaN,NaN,NaN,783.651936,798.830789


## Processing for SDT simulations

In [ ]:
# Group by model and stack params into 2d array
params = fits_sdt.groupby("model")["params"].apply(np.stack)
params_vco = params["vco"]
params_co = params["co"]
params_piece = params["piece"]

# Set upper and lower bounds on simulated params
vco_lb = params_vco.min(axis=0)
vco_ub = params_vco.max(axis=0)
co_lb = params_co.min(axis=0)
co_ub = params_co.max(axis=0)
piece_lb = params_piece.min(axis=0)
piece_ub = params_piece.max(axis=0)

# Pull out experimental parameters (same for all 16 subjects)
sdt = sdt.loc[sdt["TN"] > 106, :].reset_index(drop=True)
num_trials = len(np.unique(sdt["TN"]))
sdt_subj = sdt.loc[(sdt["SN"] == subj_id) & (sdt["session"] == "sdt"), :].reset_index(drop=True)
cols_to_copy = ["SN", "x_f", "session", "delta_xv", "vis_fb", "sigma_motor_sdt"]
simsubj = pd.DataFrame()
simsubj[cols_to_copy] = sdt_subj[cols_to_copy]
simsubj["no_fit"] = False


In [ ]:
%%time
# How many model simulations?
n_modelsims = 1

# Determine how many MC simulations to run
n_sims = 1000
n_inits = 1

# We switch off the printing from BADS
options = {"display" : 'off'}

fit_task = "sdt"

models = ["vco", "co", "piece"]
# models = ["vco", "co"]
simulated_params = []
recovered_params = []
model = []
winner_bic = []
winner_aic = []
iteration = []

# Choose subject to base simultation on (x_f)
subj_id = 1

# Initialize
resp_vco = np.zeros((n_modelsims, len(sdt_subj)))
resp_co = np.zeros((n_modelsims, len(sdt_subj)))
resp_piece = np.zeros((n_modelsims, len(sdt_subj)))

# This next loop is for both model and parameter recovery
for i in range(n_modelsims):
    print(i)
    # Generate parameter vals for each model
    vco_simparams = np.random.uniform(low=vco_lb, high=vco_ub)
    co_simparams = np.random.uniform(low=co_lb, high=co_ub)
    piece_simparams = np.random.uniform(low=piece_lb, high=piece_ub)

    # Loop through models
    for j in range(len(models)):
        print(j)
        if j == 0:
            # VCO model
            resp_vco[i, :] = vco_simulate_sdt(
                df=sdt_subj,
                C=vco_simparams[0], 
                sigma_v=vco_simparams[1], 
                b=vco_simparams[2], 
            )
            simulated_model = "vco"
            simulated_params.append(vco_simparams)
            simsubj["pert_resp"] = resp_vco[i, :]  
        elif j == 1: 
            # CO model
            resp_co[i, :] = co_simulate_sdt(
                df=sdt_subj,
                C=co_simparams[0], 
                sigma_comb=co_simparams[1],
                b=None
            )
            simulated_model = "co"
            simulated_params.append(co_simparams)
            simsubj["pert_resp"] = resp_co[i, :]
        elif j == 2:
            # PIECE model
            resp_piece[i, :] = piece_simulate_sdt(
                df=sdt_subj,
                alpha=piece_simparams[0], 
                beta=piece_simparams[1],
                sigma_p=piece_simparams[2],
                sigma_pert=piece_simparams[3],
                b=piece_simparams[4],
            )
            simulated_model = "piece"
            simulated_params.append(piece_simparams)
            simsubj["pert_resp"] = resp_piece[i, :]
            
        # Fit simulated data:
        # VCO parameter bounds (C, sigma_v, b) 
        bounds       = [(0.1, 10), (0.1, 10.0), (-5, 5)]
        plaus_bounds = [(0.5, 5), (0.5, 6), (-2.5, 2.5)]
        fits_vco, aic_vco, bic_vco = fit_bads("vco", subj_id, simsubj, bounds, plaus_bounds, n_inits=n_inits, 
                                              n_sims=n_sims, fit_task=fit_task, options=options)
        # This conditional is for parameter recovery
        if simulated_model == "vco":
            recovered_params.append(fits_vco.x)

        # CO parameter bounds (C, sigma_comb) 
        bounds = [(0.1, 10), (0.1, 10.0)]
        plaus_bounds = [(0.5, 5), (0.5, 6)]
        fits_co, aic_co, bic_co = fit_bads("co", subj_id, simsubj, bounds, plaus_bounds, n_inits=n_inits, 
                                     n_sims=n_sims, fit_task=fit_task, options=options)
        # This conditional is for parameter recovery
        if simulated_model == "co":
            recovered_params.append(fits_co.x)

        # PIECE parameter bounds: alpha, beta, sigma_p, sigma_pert, b,
        bounds       = ((0.1, 5), (0.001, 1), (0.1, 10), (0.1, 120), (-5, 5))
        plaus_bounds = ((0.5, 4), (0.01, 0.5), (0.3, 6), (3, 20), (-2.5, 2.5))
        fits_piece, aic_piece, bic_piece = fit_bads("piece", subj_id, simsubj, bounds, plaus_bounds, n_inits=n_inits, 
                                        n_sims=n_sims, fit_task=fit_task, options=options)
        # This conditional is for parameter recovery
        if simulated_model == "piece":
            recovered_params.append(fits_piece.x)

        # These lines are for model comparison/recovery: store winning model
        winner_aic.append(models[np.argmin([aic_vco, aic_co, aic_piece])])
        winner_bic.append(models[np.argmin([bic_vco, bic_co, bic_piece])])
        # winner_aic.append(models[np.argmin([aic_vco, aic_co, ])])
        # winner_bic.append(models[np.argmin([bic_vco, bic_co, ])])
        
        # Update variables to properly index in dataframe
        model.append(simulated_model)
        iteration.append(i)

# Store variables in data frame
sdt_recovery = pd.DataFrame({
    "iteration": iteration,
    "model":model, 
    "winner_aic":winner_aic, 
    "winner_bic":winner_bic,
    "simulated_params": simulated_params,
    "recovered_params": recovered_params
})
        

In [ ]:
df_model

In [ ]:
df_model.loc[2, "simulated_params"]

In [ ]:
df_model.loc[2, "recovered_params"]

---
## Processing for adaptation simulations

In [5]:
# Group by model and stack params into 2d array
params = fits_adapt.groupby("model")["params"].apply(np.stack)
params_vco = params["vco"]
params_co = params["co"]
params_piece = params["piece"]

# Set upper and lower bounds on simulated params
vco_lb = params_vco.min(axis=0)
vco_ub = params_vco.max(axis=0)
co_lb = params_co.min(axis=0)
co_ub = params_co.max(axis=0)
piece_lb = params_piece.min(axis=0)
piece_ub = params_piece.max(axis=0)

# Choose subject to base simultation on (x_f)
subj_id = 1

# Pull out experimental parameters (same for all 16 subjects)
adapt = adapt.loc[adapt["TN"] > 100, :].reset_index(drop=True)
adapt_subj = adapt.loc[(adapt["SN"] == subj_id) & (adapt["session"] == "adapt"), :].reset_index(drop=True)
cols_to_copy = ["SN", "TN", "session", "delta_xv", "vis_fb", "x_f", "sigma_motor_adapt"]
simsubj = pd.DataFrame()
simsubj[cols_to_copy] = adapt_subj[cols_to_copy]
simsubj["no_fit"] = False

In [7]:
%%time
# How many model simulations?
n_modelsims = 1

# Determine how many MC simulations to run
n_sims = 1000
n_inits = 1

# We switch off the printing from BADS
options = {"display" : 'off'}

fit_task = "adapt"

models = ["vco", "co", "piece"]
# models = ["vco", "co"]
simulated_params = []
recovered_params = []
model = []
winner_bic = []
winner_aic = []
iteration = []

resp_vco = np.zeros((n_modelsims, len(adapt_subj)))
resp_co = np.zeros((n_modelsims, len(adapt_subj)))
resp_piece = np.zeros((n_modelsims, len(adapt_subj)))

# This next loop is for both model and parameter recovery
for i in range(n_modelsims):
    print(i)
    # Generate parameter vals for each model
    vco_simparams = np.random.uniform(low=vco_lb, high=vco_ub)
    co_simparams = np.random.uniform(low=co_lb, high=co_ub)
    piece_simparams = np.random.uniform(low=piece_lb, high=piece_ub)

    # Loop through models
    for j in range(len(models)):
        print(j)
        if j == 0:
            # VCO model
            resp_vco[i, :] = vco_simulate_adapt(
                df=simsubj,
                C=vco_simparams[0], 
                B=vco_simparams[1], 
                sigma_v=vco_simparams[2], 
                b=vco_simparams[3], 
            )
            simulated_model = "vco"
            simulated_params.append(vco_simparams)
            simsubj["x_f"] = resp_vco[i, :]  
        elif j == 1: 
            # CO model
            resp_co[i, :] = co_simulate_adapt(
                df=simsubj,
                C=co_simparams[0],
                B=co_simparams[1],
                sigma_comb=co_simparams[2],
                b=co_simparams[3],
            )
            simulated_model = "co"
            simulated_params.append(co_simparams)
            simsubj["x_f"] = resp_co[i, :]
        elif j == 2:
            # PIECE model
            resp_piece[i, :] = piece_simulate_adapt(
                df=simsubj,
                alpha=piece_simparams[0], 
                beta=piece_simparams[1],
                sigma_p=piece_simparams[2],
                sigma_pert=piece_simparams[3],
                b=piece_simparams[4],
            )
            simulated_model = "piece"
            simulated_params.append(piece_simparams)
            simsubj["x_f"] = resp_piece[i, :]
            
        # Fit simulated data:
        # VCO: Parameter bounds (C, B, sigma_v, b) 
        bounds       = [(0.1, 10), (0.01, 1), (0.1, 10.0), (-5, 5)]
        plaus_bounds = [(0.5, 5), (0.05, 0.6), (0.5, 6), (-2.5, 2.5)]
        fits_vco, aic_vco, bic_vco = fit_bads("vco", subj_id, simsubj, bounds, plaus_bounds, n_inits=n_inits, 
                                              n_sims=n_sims, fit_task=fit_task, options=options)
        # This conditional is for parameter recovery
        if simulated_model == "vco":
            recovered_params.append(fits_vco.x)

        # CO parameter bounds (C, B, sigma_comb, b) 
        bounds       = [(0.1, 10), (0.01, 1), (0.1, 10.0), (-5, 5)]
        plaus_bounds = [(0.5, 5), (0.05, 0.6), (0.5, 6), (-2.5, 2.5)]
        fits_co, aic_co, bic_co = fit_bads("co", subj_id, simsubj, bounds, plaus_bounds, n_inits=n_inits, 
                                     n_sims=n_sims, fit_task=fit_task, options=options)
        # This conditional is for parameter recovery
        if simulated_model == "co":
            recovered_params.append(fits_co.x)

        # PIECE parameter bounds: alpha, beta, sigma_p, sigma_pert, b,
        bounds       = ((0.1, 5), (0.001, 1), (0.1, 10), (0.1, 120), (-5, 5))
        plaus_bounds = ((0.5, 4), (0.01, 0.5), (0.3, 6), (3, 20), (-2.5, 2.5))
        fits_piece, aic_piece, bic_piece = fit_bads("piece", subj_id, simsubj, bounds, plaus_bounds, n_inits=n_inits, 
                                        n_sims=n_sims, fit_task=fit_task, options=options)
        # This conditional is for parameter recovery
        if simulated_model == "piece":
            recovered_params.append(fits_piece.x)

        # These lines are for model comparison/recovery: store winning model
        winner_aic.append(models[np.argmin([aic_vco, aic_co, aic_piece])])
        winner_bic.append(models[np.argmin([bic_vco, bic_co, bic_piece])])
        # winner_aic.append(models[np.argmin([aic_vco, aic_co, ])])
        # winner_bic.append(models[np.argmin([bic_vco, bic_co, ])])
        
        # Update variables to properly index in dataframe
        model.append(simulated_model)
        iteration.append(i)

# Store variables in data frame
adapt_recovery = pd.DataFrame({
    "iteration": iteration,
    "model":model, 
    "winner_aic":winner_aic, 
    "winner_bic":winner_bic,
    "simulated_params": simulated_params,
    "recovered_params": recovered_params, 
})
        

0
0
Subject Number: 1
Running optimization 0...
Subject Number: 1
Running optimization 0...
Subject Number: 1
Running optimization 0...


/opt/anaconda3/envs/ige-ege/lib/python3.11/site-packages/pybads/init_functions/init_sobol.py:39: RuntimeWarning: invalid value encountered in cast
  str_seed = u0[0 : np.minimum(11, len(u0))].astype(np.uint64)


1
Subject Number: 1
Running optimization 0...


/opt/anaconda3/envs/ige-ege/lib/python3.11/site-packages/pybads/init_functions/init_sobol.py:39: RuntimeWarning: invalid value encountered in cast
  str_seed = u0[0 : np.minimum(11, len(u0))].astype(np.uint64)


Subject Number: 1
Running optimization 0...
Subject Number: 1
Running optimization 0...
2
Subject Number: 1
Running optimization 0...


/opt/anaconda3/envs/ige-ege/lib/python3.11/site-packages/pybads/init_functions/init_sobol.py:39: RuntimeWarning: invalid value encountered in cast
  str_seed = u0[0 : np.minimum(11, len(u0))].astype(np.uint64)


Subject Number: 1
Running optimization 0...
Subject Number: 1
Running optimization 0...


/opt/anaconda3/envs/ige-ege/lib/python3.11/site-packages/pybads/init_functions/init_sobol.py:39: RuntimeWarning: invalid value encountered in cast
  str_seed = u0[0 : np.minimum(11, len(u0))].astype(np.uint64)


CPU times: user 40min 18s, sys: 1h 4min 7s, total: 1h 44min 26s
Wall time: 13min 56s


In [18]:
adapt_recovery

,iteration,model,winner_aic,winner_bic,simulated_params,recovered_params
0,0,vco,vco,vco,"[8.281425855490976, 0.21623166356251305, 3.013...","[7.817854811390366, 0.21846746390495897, 2.579..."
1,0,co,co,co,"[3.5602379420207813, 0.3730923209953924, 3.984...","[3.121597738246415, 0.36264066869378764, 4.424..."
2,0,piece,piece,piece,"[0.18671502900502254, 0.18713986668166277, 0.5...","[0.5118677616119385, 0.15318486806429732, 0.10..."


In [21]:
np.round(adapt_recovery.loc[2, "simulated_params"], 5)

array([ 0.18672,  0.18714,  0.57881, 63.36131, -0.82694])

In [22]:
np.round(adapt_recovery.loc[2, "recovered_params"], 5)

array([ 0.51187,  0.15318,  0.1    , 69.87236, -0.81959])